# EduCrew Studio: free AI episode maker
**Series:** EduCrew: From a Little Dream to a Bigger World (10 episodes, 3D animated, 9:16, 60 s)

**Settings (once):** Accelerator **GPU T4 x2** · Internet **on** · Persistence **Files only** · Add-ons → Secrets: `HF_TOKEN` ticked.

### Every episode: 3 cells, run them in order (click inside, Shift + Enter)
1. **Setup**: set `EPISODE`, run.
2. **Pictures** (about 10 min): look at them. Redo any weak one (see the bottom), then go on.
3. **Everything else** (about 1.5 hours): clips on both GPUs, voices, sounds, final Reel. You can switch tabs; keep the Mac awake.

Make the episodes **in order** (01 → 10). If anything stops, run the same cell again: finished parts are kept.

In [ ]:
# ▶ WHICH EPISODE (change only the number)
EPISODE = "episodes/EduCrew_Biography_Episode_01.json"

import glob, os, shutil
PROJECT = "/kaggle/working/educrew-studio"
if not os.path.exists(PROJECT):
    !git clone -q https://github.com/educatorscircleec-dot/educrew-studio {PROJECT}
!cd {PROJECT} && git fetch -q origin && git reset -q --hard origin/main && git log --oneline -1
os.chdir(PROJECT)
NAME = os.path.splitext(os.path.basename(EPISODE))[0]
WORK = f"work/{NAME}"
for z in glob.glob("/kaggle/input/**/work.zip", recursive=True):
    shutil.unpack_archive(z, PROJECT); print("restored", z)
try:
    import compel, psutil
except ImportError:
    !pip install -q -U diffusers transformers accelerate ftfy imageio-ffmpeg sentencepiece "compel>=2.5" psutil
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))
from IPython.display import display, Image, Video, Audio
def show(kind):
    """show("keyframes") · show("clips") · show("voice") · show("sfx")"""
    ext = {"keyframes": "png", "clips": "mp4", "voice": "wav", "sfx": "wav"}[kind]
    for p in sorted(glob.glob(f"{WORK}/{kind}/*.{ext}")):
        print(p); display(Image(p, width=220) if ext == "png" else Video(p, embed=True, width=220) if ext == "mp4" else Audio(p))
print("Ready:", NAME)

## 2. Pictures (about 10 min). Check them before going on.

In [ ]:
!python tools/make_episode.py $EPISODE --only pictures
show("keyframes")

## 3. Everything else (about 1.5 hours): clips → voices → sounds → final Reel
Download the Reel from **Output → educrew-studio → output**.

In [ ]:
!python tools/make_episode.py $EPISODE --skip pictures
reel = glob.glob(f"output/*{NAME}.mp4")
Video(reel[0], embed=True, width=300) if reel else print("No Reel yet; see the message above.")

## Redo one shot (only if needed)
Remove the `#`, set the shot number, and run. This remakes that shot's **picture, clip and sound**, then rebuilds the Reel (about 15 min).
A different `--seed` gives a different version. Voices: use the second line.

In [ ]:
# !python tools/make_episode.py $EPISODE --redo 1 --seed 7
# !cd tools && /kaggle/working/.tts_env/bin/python make_voices.py ../$EPISODE --shots 5 --force --seed 21 && cd .. && python assemble.py $EPISODE

## Look at the parts (optional)

In [ ]:
# show("clips")
# show("voice")
# show("sfx")

## Save progress
Only if Persistence is **not** on: run this, then download `work.zip` from **Output**.

In [ ]:
shutil.make_archive("/kaggle/working/work", "zip", PROJECT, "work")
print("saved /kaggle/working/work.zip")